# INFORME TÉCNICO: SISTEMA DE RECUPERACIÓN DE INFORMACIÓN (VACANTES LABORALES)
**Asignatura:** Recuperación de Información  
**Integrantes:** Bravo Leandro - Enríquez Michael - Ochoa Aubertin  
**Fecha:** 1 de junio de 2026  

---

## 1. Introducción y Descripción del Corpus

El presente proyecto implementa y evalúa un Sistema de Recuperación de Información enfocado en un corpus masivo de ofertas de empleo orientadas a perfiles de ingeniería y administración. La colección de datos original se encuentra segmentada en **24 archivos en formato CSV**, donde cada archivo agrupa las vacantes correspondientes a una disciplina académica específica (por ejemplo, *Ciencia_de_Datos_Merged.csv*, *Software_Merged.csv*, *Mecatrónica_Merged.csv*, entre otros).

Al integrar la totalidad de las fuentes mediante un proceso de concatenación directa en la fase de carga de datos, el corpus definitivo alcanza un tamaño de **75,695 documentos**. Cada registro representa una oferta laboral individual. Con la finalidad de optimizar la eficiencia computacional y enfocar el sistema en la recuperación por contenido textual, se aplicó un filtro de proyección seleccionando exclusivamente las siguientes variables esenciales:
* `job_id`: Identificador único (hash MD5/SHA256) utilizado para control de redundancia y asignación de índices.
* `job_title`: Título formal de la vacante.
* `company`: Organización ofertante.
* `careers_required`: Vector de carreras universitarias validadas.
* `text`: Derivado directamente del campo de descripciones limpias (`description_final`), el cual contiene las responsabilidades, competencias y requisitos detallados del puesto de trabajo.

Un análisis estadístico descriptivo post-procesamiento determinó que los documentos presentan una longitud promedio de **60 tokens válidos**, registrando un volumen máximo aislado de 1,582 tokens y casos mínimos de documentos vacíos (0 tokens) tras la remoción estricta de elementos sintácticos no informativos.

## Paso 1: Librerías y Configuración

In [1]:
import os
import pandas as pd
import sys
from pathlib import Path
current_dir = Path.cwd()
from collections import Counter, defaultdict
from sklearn.metrics.pairwise import cosine_similarity

# Configurar rutas de importación para acceder a los módulos en sources
sources_dir = current_dir / 'sources'
if str(sources_dir) not in sys.path:
    sys.path.insert(0, str(sources_dir))

from prepro_func import ensure_nltk_resources, remove_special_characters, tokenize, stemming_tokens, build_tfidf_matrix, score_queries_tfidf
from bm25_model import build_bm25_index, bm25_score_doc, bm25_rank, score_queries_bm25
from jaccard_similarity import jaccard_similarity, binary_vector_jaccard

ensure_nltk_resources()

print("Librerías y módulos cargados correctamente")

Librerías y módulos cargados correctamente


## Paso 2: Carga de archivos y construcción del corpus

In [2]:
data_dir = Path.cwd() / 'data'
csv_files = sorted(data_dir.glob('*.csv'))

print(f"Archivos CSV encontrados: {len(csv_files)}")

# Cargar todos los CSVs
dataframes = []
for csv_file in csv_files:
    try:
        df = pd.read_csv(csv_file, encoding='utf-8')
        print(f"✅ {csv_file.name}: {len(df)} filas cargadas")
        dataframes.append(df)
    except Exception as e:
        print(f"❌ Error cargando {csv_file.name}: {e}")

# Concatenar todos los dataframes
df_corpus = pd.concat(dataframes, ignore_index=True)

print(f"\nCorpus completo: {len(df_corpus)} documentos")
print(f"Columnas disponibles: {df_corpus.columns.tolist()}")

# Seleccionar columnas relevantes y usar description_final como texto principal
df_corpus = df_corpus[['job_id', 'job_title', 'company', 'careers_required', 'description_final']].copy()
df_corpus.rename(columns={'description_final': 'text'}, inplace=True)

# Verificar datos
print(f"\nCorpus preparado con columnas: {df_corpus.columns.tolist()}")
print(f"Primeras 2 filas (primeros 200 caracteres):")
df_corpus.head(2)

Archivos CSV encontrados: 24
✅ Administración_de_Empresas_Merged.csv: 5220 filas cargadas
✅ Agroindustria_Merged.csv: 1688 filas cargadas
✅ Ciencia_de_Datos_Merged.csv: 4836 filas cargadas
✅ Computación_Merged.csv: 4101 filas cargadas
✅ Economía_Merged.csv: 2506 filas cargadas
✅ Electricidad_Merged.csv: 2066 filas cargadas
✅ Electrónica_y_Automatización_Merged.csv: 6349 filas cargadas
✅ Física_Merged.csv: 1614 filas cargadas
✅ Geología_Merged.csv: 1074 filas cargadas
✅ Ingeniería_Ambiental_Merged.csv: 3295 filas cargadas
✅ Ingeniería_Civil_Merged.csv: 3864 filas cargadas
✅ Ingeniería_de_la_Producción_Merged.csv: 4503 filas cargadas
✅ Ingeniería_Química_Merged.csv: 1148 filas cargadas
✅ Inteligencia_Artificial_Merged.csv: 6805 filas cargadas
✅ Matemática_Aplicada_Merged.csv: 383 filas cargadas
✅ Matemática_Merged.csv: 399 filas cargadas
✅ Materiales_Merged.csv: 2127 filas cargadas
✅ Mecatrónica_Merged.csv: 1338 filas cargadas
✅ Mecánica_Merged.csv: 1633 filas cargadas
✅ Petróleos_Merged

,job_id,job_title,company,careers_required,text
0,69fc771c82ad86bd1f49714ab4c488afbf817a99d5702f...,Asistente de Administración y Gerencia experie...,HIALPESA,[],Importante empresa Textil con más de 40 años e...
1,4791ac695898b5787d37a855ad4409f4786898125a537c...,Docente JP Administración de empresas Tumbes,SENATI,[],DOCENTE EN ADMINISTRACIÓN Institución Líder y ...


## 2. Decisiones de Diseño y Pipeline de Preprocesamiento

El sistema se diseñó bajo una arquitectura modular en Python, separando las etapas de preprocesamiento, indexación por términos, vectorización semántica densa y evaluación. Para transformar el texto plano original en una representación lógica apta para los modelos basados en términos, se implementó un pipeline secuencial:

1. **Limpieza de Caracteres Especiales:** Filtrado mediante expresiones regulares para eliminar etiquetas remanentes de HTML (como `&nbsp;`), signos de puntuación y caracteres no alfanuméricos, forzando la conversión a minúsculas para unificar el espacio de características.
2. **Tokenización y Remoción de Stopwords:** Segmentación de cadenas en listas de palabras utilizando los recursos léxicos en español de la librería `NLTK`. Se descartaron artículos, preposiciones y conectores (stopwords) dado que exhiben una alta frecuencia de aparición pero carecen de carga semántica discriminatoria.
3. **Stemming (Reducción Léxica):** Aplicación del algoritmo *Snowball Stemmer* para el idioma español. Este proceso reduce las palabras a sus raíces o lemas morfológicos (por ejemplo, los tokens `'importante'`, `'dedicada'` y `'fabricación'` se reducen a `'import'`, `'dedic'` y `'fabric'`). Con esto se reduce la dimensionalidad del vocabulario global y se agrupan variantes gramaticales bajo un mismo término.

## Paso 3: Limpieza de caracteres especiales

In [3]:
# Mostrar antes de limpiar
print("ANTES (200 caracteres):")
print(df_corpus['text'].iloc[0][:200]+"\n")

# Aplicar remove_special_characters
df_corpus['clean_text'] = df_corpus['text'].fillna('').apply(remove_special_characters)

print("DESPUÉS (200 caracteres):")
print(df_corpus['clean_text'].iloc[0][:200])
print("\n" + "="*80 + "\n")

# Estadísticas
print(f"✅ Limpieza completada")
print(f"Promedio caracteres antes: {df_corpus['text'].str.len().mean():.0f}")
print(f"Promedio caracteres después: {df_corpus['clean_text'].str.len().mean():.0f}")
print(f"\nMuestra de 3 documentos limpios:")
df_corpus[['job_title', 'clean_text']].head(3)

ANTES (200 caracteres):
Importante empresa Textil con más de 40 años en el mercado, dedicada a la fabricación y exportación de prendas de vestir se encuentra en busca de ASISTENTE ADMINISTRATIVO FINANCIERO Requisitos Bachill

DESPUÉS (200 caracteres):
Importante empresa Textil con más de 40 años en el mercado dedicada a la fabricación y exportación de prendas de vestir se encuentra en busca de ASISTENTE ADMINISTRATIVO FINANCIERO Requisitos Bachille


✅ Limpieza completada
Promedio caracteres antes: 714
Promedio caracteres después: 687

Muestra de 3 documentos limpios:


,job_title,clean_text
0,Asistente de Administración y Gerencia experie...,Importante empresa Textil con más de 40 años e...
1,Docente JP Administración de empresas Tumbes,DOCENTE EN ADMINISTRACIÓN Institución Líder y ...
2,Aprendiz Universitario administración de empr...,Nos encontramos en la búsqueda de un aprendiz ...


## Paso 4: Tokenización

In [4]:
# Aplicar tokenización en español
df_corpus['tokens'] = df_corpus['clean_text'].apply(
    lambda x: tokenize(x, language='spanish', remove_stopwords=True)
)

# Estadísticas
print(f"✅ Tokenización completada")
print(f"Promedio de tokens por documento: {df_corpus['tokens'].apply(len).mean():.0f}")
print(f"Máximo número de tokens: {df_corpus['tokens'].apply(len).max()}")
print(f"Mínimo número de tokens: {df_corpus['tokens'].apply(len).min()}")
print("\n")

print("Muestra de 3 documentos tokenizados:")
for i in range(3):
    print(f"\n Documento {i+1}: {df_corpus['job_title'].iloc[i]}")
    tokens_list = df_corpus['tokens'].iloc[i]
    print(f"   Primeros 15 tokens: {tokens_list[:15]}")
    print(f"   Total de tokens: {len(tokens_list)}")

✅ Tokenización completada
Promedio de tokens por documento: 60
Máximo número de tokens: 1582
Mínimo número de tokens: 0


Muestra de 3 documentos tokenizados:

 Documento 1: Asistente de Administración y Gerencia experiencia en empresas industriales
   Primeros 15 tokens: ['importante', 'empresa', 'textil', 'años', 'mercado', 'dedicada', 'fabricación', 'exportación', 'prendas', 'vestir', 'encuentra', 'busca', 'asistente', 'administrativo', 'financiero']
   Total de tokens: 78

 Documento 2: Docente JP Administración  de empresas Tumbes
   Primeros 15 tokens: ['docente', 'administración', 'institución', 'líder', 'prestigiosa', 'formación', 'profesional', 'apoya', 'industria', 'nacional', 'contexto', 'global', 'contribuye', 'mejora', 'calidad']
   Total de tokens: 158

 Documento 3: Aprendiz  Universitario administración de empresas  Etapa práctica
   Primeros 15 tokens: ['encontramos', 'búsqueda', 'aprendiz', 'estudiante', 'universitario', 'administración', 'empresas', 'convenio', 'sena

## Paso 5: Stemming 

In [5]:
# Aplicar stemming a los tokens
df_corpus['stemmed'] = df_corpus['tokens'].apply(
    lambda tokens: ' '.join(stemming_tokens(tokens, language='spanish'))
)

print(f"✅ Stemming completado")
print(f"Promedio de caracteres en texto stemmed: {df_corpus['stemmed'].str.len().mean():.0f}")

print("\n")

print("Comparación Tokens → Stemmed (3 ejemplos):\n")
for i in range(3):
    print(f" Documento {i+1}: {df_corpus['job_title'].iloc[i]}")
    tokens_list = df_corpus['tokens'].iloc[i][:10]
    print(f"   Tokens (primeros 10):  {tokens_list}")
    stemmed_list = df_corpus['stemmed'].iloc[i].split()[:10]
    print(f"   Stemmed (primeros 10): {stemmed_list}")
    print()

✅ Stemming completado
Promedio de caracteres en texto stemmed: 422


Comparación Tokens → Stemmed (3 ejemplos):

 Documento 1: Asistente de Administración y Gerencia experiencia en empresas industriales
   Tokens (primeros 10):  ['importante', 'empresa', 'textil', 'años', 'mercado', 'dedicada', 'fabricación', 'exportación', 'prendas', 'vestir']
   Stemmed (primeros 10): ['import', 'empres', 'textil', 'años', 'merc', 'dedic', 'fabric', 'export', 'prend', 'vest']

 Documento 2: Docente JP Administración  de empresas Tumbes
   Tokens (primeros 10):  ['docente', 'administración', 'institución', 'líder', 'prestigiosa', 'formación', 'profesional', 'apoya', 'industria', 'nacional']
   Stemmed (primeros 10): ['docent', 'administr', 'institu', 'lid', 'prestigi', 'formacion', 'profesional', 'apoy', 'industri', 'nacional']

 Documento 3: Aprendiz  Universitario administración de empresas  Etapa práctica
   Tokens (primeros 10):  ['encontramos', 'búsqueda', 'aprendiz', 'estudiante', 'universitario

## 3. Estructuras de Almacenamiento e Indexación

Para dar soporte a los diferentes modelos tradicionales de recuperación, se construyeron estructuras de datos persistentes y matrices optimizadas para el manejo de la dispersión de datos:

* **Índice Invertido:** Construido desde cero mapeando cada término único del vocabulario hacia una estructura de *postings list*. Cada elemento de la lista registra el índice del documento, el identificador (`job_id`) y la frecuencia de término interna ($tf$), permitiendo resolver de forma eficiente modelos de conteo y consultas basadas en texto libre. El vocabulario resultante alcanzó un total de **83,775 términos únicos**.
* **Matriz TF-IDF:** Construida sobre los strings reducidos por stemming. Genera un espacio vectorial disperso de dimensiones $75,695 \times 56,999$ (términos significativos), registrando un porcentaje de **sparsidad del 99.92%**, lo cual justifica el uso de almacenamiento matricial comprimido.
* **Base de Datos Vectorial (ChromaDB):** Para dar soporte al modelo de recuperación semántica, se instanció un motor persistente local de `ChromaDB`. Los documentos limpios se codificaron en vectores densos utilizando el modelo preentrenado `all-MiniLM-L6-v2` de la suite `SentenceTransformers`. La inserción de los 75,695 registros se estructuró a través de un esquema de carga por lotes (*batching*) de 5,000 elementos para mitigar el desbordamiento de la memoria RAM del sistema.

## Paso 6: Índice invertido

In [6]:
# Construcción
df_corpus['tokens'] = df_corpus['tokens'].apply(lambda x: x if isinstance(x, list) else [])

inverted_index = defaultdict(lambda: {
    'doc_freq': 0,
    'total_freq': 0,
    'postings': []
})

for doc_idx, tokens in enumerate(df_corpus['tokens']):
    term_counts = Counter(tokens)
    for term, freq in term_counts.items():
        entry = inverted_index[term]
        entry['doc_freq'] += 1
        entry['total_freq'] += freq
        entry['postings'].append({
            'doc_index': doc_idx,
            'job_id': df_corpus['job_id'].iloc[doc_idx],
            'job_title': df_corpus['job_title'].iloc[doc_idx],
            'freq': freq
        })

# Convertir a diccionario normal para facilitar inspección
inverted_index = dict(inverted_index)

# Mostrar información general del índice
print(f"✅ Índice invertido construido")
print(f"Términos únicos en el índice: {len(inverted_index)}")

# Crear un DataFrame del índice invertido para mostrar estadísticas de los términos
summary_rows = []
for term in sorted(inverted_index):
    summary_rows.append({
        'term': term,
        'doc_freq': inverted_index[term]['doc_freq'],
        'total_freq': inverted_index[term]['total_freq']
    })
summary_df = pd.DataFrame(summary_rows)
print("\nÍndice invertido:")
display(summary_df)

# Mostrar el índice invertido de algunos términos de ejemplo
test_terms = list(summary_df['term'].head(5))
for term in test_terms:
    print(f"\nTérmino: '{term}'")
    print(f"  Documentos: {inverted_index[term]['doc_freq']}")
    print(f"  Frecuencia total: {inverted_index[term]['total_freq']}")
    print("  Posting list:")
    for posting in inverted_index[term]['postings'][:5]:
        print(f"    - doc_index={posting['doc_index']}, job_id={posting['job_id']}, freq={posting['freq']}, title={posting['job_title']}")


✅ Índice invertido construido
Términos únicos en el índice: 83775

Índice invertido:


,term,doc_freq,total_freq
0,aa,34,37
1,aaa,25,32
2,aaaradius,1,1
3,aaas,1,1
4,aace,1,1
...,...,...,...
83770,úpo,1,1
83771,úselos,2,2
83772,úsqueda,1,1
83773,útil,174,195



Término: 'aa'
  Documentos: 34
  Frecuencia total: 37
  Posting list:
    - doc_index=14422, job_id=3f39df6adf74b1119ced57864fba75982b8c246723024ec1788774112c9a7d23, freq=1, title=Laborer
    - doc_index=15849, job_id=2ab4ada1512f59548c43f1d3823693c803fd38979f802e35b068f33e64f29982, freq=1, title=LAUREATA\\O IN GIURISPRUDENZA/ECONOMIA o CONSULENTE D' IMPRESA
    - doc_index=18356, job_id=5f46afc2a364f40917cbb54b034c733501d49a54b16a14e445092a645a11b0c6, freq=1, title=T.S.U. en Electricidad
    - doc_index=21420, job_id=6dfd598d8f2efac8963923068a292d26c3bc60bbe00d4955949bcf5c54103216, freq=1, title=Laborer
    - doc_index=21431, job_id=c71e17a9e97a6d9cc6d773517987cb8bd0896c45fd3f63bf2c777cb6436e67f2, freq=1, title=Cement Driver

Término: 'aaa'
  Documentos: 25
  Frecuencia total: 32
  Posting list:
    - doc_index=7467, job_id=bda92e9fbaab5d71114392743bb6c82bdf5d75a437613f98f0ef48d9dcd4c603, freq=1, title=Pastry Chef de Partie
    - doc_index=9000, job_id=aa25238b7ed6a546b0cbc1cfe8bf359

## Paso 7: TF-IDF

In [7]:
# Construir matriz TF-IDF a partir del texto stemmed
vectorizer, tfidf_matrix = build_tfidf_matrix(df_corpus['stemmed'].tolist())

print(f"✅ Matriz TF-IDF construida")
print(f"   Dimensiones: {tfidf_matrix.shape[0]} documentos × {tfidf_matrix.shape[1]} términos")
print(f"   Sparsidad: {(1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])) * 100:.2f}%")

# Obtener los términos más frecuentes
feature_names = vectorizer.get_feature_names_out()
print(f"\n Primeros 20 términos del vocabulario:")
print(f"   {list(feature_names[:20])}")

# Estadísticas por documento
doc_term_counts = (tfidf_matrix > 0).sum(axis=1).A1  # Contar términos no-cero por documento
print(f"\n Estadísticas de términos por documento:")
print(f"   Promedio de términos únicos: {doc_term_counts.mean():.0f}")
print(f"   Máximo: {doc_term_counts.max()}")
print(f"   Mínimo: {doc_term_counts.min()}")

print(f"\n✅ df_corpus tiene {len(df_corpus)} documentos listos para recuperación")
print(f"\nEstructura final de df_corpus:")
print(df_corpus.columns.tolist())

✅ Matriz TF-IDF construida
   Dimensiones: 75695 documentos × 56999 términos
   Sparsidad: 99.92%

 Primeros 20 términos del vocabulario:
   ['aa', 'aaa', 'aaaradius', 'aaas', 'aac', 'aacp', 'aacsb', 'aact', 'aad', 'aadus', 'aae', 'aaeeo', 'aaeeoveteransdiscapacit', 'aaf', 'aah', 'aai', 'aailndi', 'aailndiam', 'aajax', 'aan']

 Estadísticas de términos por documento:
   Promedio de términos únicos: 44
   Máximo: 708
   Mínimo: 0

✅ df_corpus tiene 75695 documentos listos para recuperación

Estructura final de df_corpus:
['job_id', 'job_title', 'company', 'careers_required', 'text', 'clean_text', 'tokens', 'stemmed']


## Paso 7.1: Ejemplo de consultas TF-IDF con resultados numéricos

In [8]:
queries = [
    'inteligencia artificial',
    'ingeniería de software',
    'energía renovable',
    'análisis de datos',
    'diseño de sistemas electrónicos',
    'gestión de proyectos'
]

scores = score_queries_tfidf(vectorizer, tfidf_matrix, queries)

results = []
for query_idx, query in enumerate(queries):
    for doc_idx, score in enumerate(scores):
        results.append({
            'query': query,
            'doc_index': doc_idx,
            'score': score[query_idx],
            'job_title': df_corpus['job_title'].iloc[doc_idx],
            'company': df_corpus['company'].iloc[doc_idx],
            'preview': df_corpus['clean_text'].iloc[doc_idx][:120].replace('\n', ' ')
        })

results_df = pd.DataFrame(results)

# Mostrar top 3 documentos por consulta
for query in queries:
    print(f"\nConsulta: '{query}'")
    top_docs = results_df[results_df['query'] == query].nlargest(3, 'score')
    display(top_docs[['score', 'job_title', 'company', 'preview']])



Consulta: 'inteligencia artificial'


,score,job_title,company,preview
43621,0.474468,"Artificial Intelligence Sales Specialist III, ...",Google,s Grado o experiencia práctica equivalente 10...
47933,0.469508,Technology Analyst 2-Artificial Intelligence (...,StateJobsNY,Tecnology Analyst 2 inteligencia artificial Ba...
43441,0.449351,Strategic Artificial Intelligence Consultant -...,Business and Technology Solutions,Nuestro cliente en Rhode Island está buscando ...



Consulta: 'ingeniería de software'


,score,job_title,company,preview
124981,0.232339,Líder de equipo,Buffalo Wild Wings GO,miembros del equipo según lo solicitado Propor...
94292,0.209075,HVAC Heating and Cooling Systems Installer,Rellaire Smart Home Systems,Ya sea que recién esté comenzando o sea un ins...
119242,0.200060,Sr. Dir. Product Management - Artificial Intel...,UKG (Ultimate Kronos Group),No puedo esperar para apoyar lo que te dé un p...



Consulta: 'energía renovable'


...

,score,job_title,company,preview
276371,0.391171,Líder de equipo,Buffalo Wild Wings GO,miembros del equipo según lo solicitado Propor...
245682,0.352004,HVAC Heating and Cooling Systems Installer,Rellaire Smart Home Systems,Ya sea que recién esté comenzando o sea un ins...
270632,0.336826,Sr. Dir. Product Management - Artificial Intel...,UKG (Ultimate Kronos Group),No puedo esperar para apoyar lo que te dé un p...



Consulta: 'diseño de sistemas electrónicos'


,score,job_title,company,preview
352066,0.391171,Líder de equipo,Buffalo Wild Wings GO,miembros del equipo según lo solicitado Propor...
321377,0.352004,HVAC Heating and Cooling Systems Installer,Rellaire Smart Home Systems,Ya sea que recién esté comenzando o sea un ins...
346327,0.336826,Sr. Dir. Product Management - Artificial Intel...,UKG (Ultimate Kronos Group),No puedo esperar para apoyar lo que te dé un p...



Consulta: 'gestión de proyectos'


,score,job_title,company,preview
427761,0.391171,Líder de equipo,Buffalo Wild Wings GO,miembros del equipo según lo solicitado Propor...
397072,0.352004,HVAC Heating and Cooling Systems Installer,Rellaire Smart Home Systems,Ya sea que recién esté comenzando o sea un ins...
422022,0.336826,Sr. Dir. Product Management - Artificial Intel...,UKG (Ultimate Kronos Group),No puedo esperar para apoyar lo que te dé un p...


## Paso 8: Jaccard

In [9]:
# Usamos los tokens ya preprocesados para calcular similitud Jaccard en español

def score_queries_jaccard_tokens(queries, doc_tokens, tokenize_func=None):
    if tokenize_func is None:
        tokenize_func = tokenize

    rows = []
    for query in queries:
        query_tokens = tokenize_func(query, language='spanish', remove_stopwords=True)
        query_set = set(query_tokens)

        for doc_idx, tokens in enumerate(doc_tokens):
            score = jaccard_similarity(set(tokens), query_set)
            rows.append({
                'query': query,
                'doc_index': doc_idx,
                'score': score,
            })

    results_df = pd.DataFrame(rows)
    results_df = results_df.sort_values(by=['query', 'score'], ascending=[True, False]).reset_index(drop=True)
    return results_df

queries = [
    'inteligencia artificial',
    'ingeniería de software',
    'energía renovable',
    'análisis de datos',
    'diseño de sistemas electrónicos',
    'gestión de proyectos'
]

jaccard_results = score_queries_jaccard_tokens(queries, df_corpus['tokens'])

print(f"✅ Se calcularon puntuaciones Jaccard para {len(queries)} queries")

for query in queries:
    print(f"\nConsulta: '{query}'")
    top_docs = jaccard_results[jaccard_results['query'] == query].head(5)
    top_docs = top_docs.merge(
        df_corpus[['job_title', 'company', 'clean_text']],
        left_on='doc_index',
        right_index=True
    )
    top_docs['preview'] = top_docs['clean_text'].str[:120].str.replace('\n', ' ')
    display(top_docs[['score', 'job_title', 'company', 'preview']])

✅ Se calcularon puntuaciones Jaccard para 6 queries

Consulta: 'inteligencia artificial'


,score,job_title,company,preview
378475,0.400000,"Artificial Intelligence Analyst - Northern, VA...",Synertex LLC,Descripción del trabajo Descripción del trabaj...
378476,0.333333,AI Strategy Product Owner,Right Seat,salario beneficios sobre los derechos artific...
378477,0.250000,"Director or Regulatory, Software and Artificia...",Philips,El director o la inteligencia reguladora de so...
378478,0.250000,"Director or Regulatory, Software and Artificia...",Philips,El director o la inteligencia reguladora de so...
378479,0.222222,Researcher: Machine Learning/Artificial Intell...,National Renewable Energy Laboratory,Investigador Aprendizaje automáticoaplicacione...



Consulta: 'ingeniería de software'


,score,job_title,company,preview
302780,0.133333,Profesor(a) BA Tecnología Info. - Análisis y D...,NUC University,Bachillerato en Tecnología de Información con ...
302781,0.133333,IT Solutions Tech Lead (Staff Fullstack SWE),Databricks,Increíble crecimiento Informará al director de...
302782,0.125000,Profesor(a) BA Tecnología Info. - Análisis y D...,NUC University,con Concentración en Análisis y Desarrollo de ...
302783,0.125000,Digital Manufacturing Engineer,Belcan,ronautics o eso 3 a 5 años de experiencia en i...
302784,0.125000,Data Scientist Level 3,IntelliGenesis,Aplicación de modelos lineales Gestión de dato...



Consulta: 'energía renovable'


...

,score,job_title,company,preview
0,0.222222,Senior Data Engineer - Analytics,NaN,Únase para solicitar el papel de datos senior ...
1,0.133333,"Analytics Data Engineer, Applied Engineering",NaN,Ingeniero de datos de análisis ingeniería apli...
2,0.125000,Ingénieur devOps junior,Hs Mittweida,Gique en la transformación digital 2 Qué parti...
3,0.125000,Alternant - Data Steward / Data Analyst,Allergan,Confiabilidad calidad y análisis de datos para...
4,0.125000,United States Postal Service (USPS) Incumbent ...,General Dynamics Information Technology,Support Specialist Cloud Developer Cloud Proje...



Consulta: 'diseño de sistemas electrónicos'


,score,job_title,company,preview
75695,0.166667,Software Engineer,OMS Group,Diseño de software PLC para sistemas de automa...
75696,0.120000,Instrument Maintenance Tech,Knowhirematch,Reparación diseño ajuste calibración y solució...
75697,0.111111,Senior Systems Engineer,Inspire Medical Systems Inc.,Consideraciones de sistemas para la experienci...
75698,0.111111,ERP Anwendungsbetreuer*in (m/w/d) für SAP - 07/25,Fbh Berlin,Cadena de valor desde el diseño hasta los sist...
75699,0.105263,Solution Architect,Information Sharing Company,Diseño y diseño de aplicaciones empresariales ...



Consulta: 'gestión de proyectos'


,score,job_title,company,preview
227085,0.222222,"Manager, Project Manager, Extended Servicing| ...",Capital One,146100 166700 para el gerente gestión de pro...
227086,0.200000,Sistemista Network,Innovaway,Realización de técnicas de gestión de proyecto...
227087,0.142857,Ingénieur de projet / Project Engineer,Textron,Ingeniero de proyectos Descripción del ingeni...
227088,0.142857,Manager de projets nucléaires,Scalian,Durance teniendo una profundidad de experienci...
227089,0.142857,Technical Account Manager Réseaux/Sécurité H/F,SPIEgroup,Ingeniero Comercial París Contrato Duración de...


## Paso 8.1: Pruebas de Jaccard

In [10]:
# Pruebas básicas de la implementación de Jaccard
assert binary_vector_jaccard(['a', 'b', 'c'], ['b', 'c', 'd']) == 0.5
assert binary_vector_jaccard([], ['a', 'b']) == 0.0
assert binary_vector_jaccard(['a', 'b'], []) == 0.0
assert binary_vector_jaccard(['a', 'b'], ['a', 'b']) == 1.0

# Verificar que todas las puntuaciones estén en el rango correcto
assert jaccard_results['score'].between(0.0, 1.0).all()

# Consulta con token claramente inexistente para validar cero coincidencias
zero_query = 'token_unico_12345_xyz'
zero_scores = score_queries_jaccard_tokens([zero_query], df_corpus['tokens'])
assert zero_scores['score'].max() == 0.0

print('✅ Todas las pruebas de Jaccard pasaron correctamente')

✅ Todas las pruebas de Jaccard pasaron correctamente


## Paso 9: BM25

In [11]:
# Construir y usar BM25 sobre el corpus limpio
bm25_docs = df_corpus['clean_text'].fillna('').tolist()
bm25_index = build_bm25_index(bm25_docs)

bm25_results = score_queries_bm25(queries, bm25_docs, index=bm25_index)

print(f"✅ BM25 index construido para {bm25_index['N']} documentos")

for query in queries:
    print(f"\nConsulta BM25: '{query}'")
    top_docs = bm25_results[bm25_results['query'] == query].head(5)
    top_docs = top_docs.merge(
        df_corpus[['job_title', 'company', 'clean_text']],
        left_on='doc_index',
        right_index=True
    )
    top_docs['preview'] = top_docs['clean_text'].str[:120].str.replace('\n', ' ')
    display(top_docs[['score', 'job_title', 'company', 'preview']])

✅ BM25 index construido para 75695 documentos

Consulta BM25: 'inteligencia artificial'


,score,job_title,company,preview
378475,12.724654,Artificial Intelligence (AI) (Architect),"ActioNet, Inc.",Inteligencia artificial AI Arquitecto Unirse p...
378476,12.610989,Generative Artificial Intelligence Architect,Rite,Arquitecto de inteligencia artificial generati...
378477,12.399769,"Director, Data and Artificial Intelligence",Simpson Strong-Tie,Director Datos e Inteligencia Artificial se un...
378478,12.275137,Artificial Intelligence for Video Compression ...,Qualcomm,Inteligencia artificial para la compresión de ...
378479,12.204157,Artificial Intelligence Attorney - Vice Presid...,JPMorganChase,Abogado de inteligencia artificial Vicepresid...



Consulta BM25: 'ingeniería de software'


,score,job_title,company,preview
302780,9.094919,"Senior Manager, Software Engineering, Back End...",Capital One,Gerente Senior Ingeniería de Software Back End...
302781,9.094919,"Senior Manager, Software Engineering, Back End...",Capital One,Gerente Senior Ingeniería de Software Back End...
302782,9.094919,"Senior Manager, Software Engineering, Back End...",Capital One,Gerente Senior Ingeniería de Software Back End...
302783,8.972164,"Senior Director, Software Engineering",Capital One,Descripción general Wework Reforma Latino 9700...
302784,8.949289,M - 3/24 - 756305 - Sr. Cloud Engineer,Focused HR Solutions,Pango de entrega que incluye y no se limita al...



Consulta BM25: 'energía renovable'


...

,score,job_title,company,preview
0,10.203751,Alteryx Lead Consultant - Migration,Tiger Analytics,Tiger Analytics es una firma de consultoría de...
1,10.203751,Alteryx Lead Consultant - Migration,Tiger Analytics,Tiger Analytics es una firma de consultoría de...
2,10.090111,Data Scientist in Washington DC - TS/SCI Clear...,Maania Consultancy Services,Habilidades requeridas Análisis de datos Cie...
3,9.938990,"Principal Data Scientist, Artificial Intellige...",BMO U.S.,El científico principal de datos la inteligenc...
4,9.783221,Sr Data Scientist with Fraud Detection & GenAI...,Simple Solutions,Antecedentes educativos como mínimo de una mae...



Consulta BM25: 'diseño de sistemas electrónicos'


,score,job_title,company,preview
75695,14.115959,Instrument Maintenance Tech,Knowhirematch,Reparación diseño ajuste calibración y solució...
75696,14.028534,Electrical Engineer - Robotics / Automation / ...,USA Tech Recruit,Estamos trabajando con una emocionante startup...
75697,13.305483,Ingeniero de electrónica / telecomunicaciones,"ANSI, Análisis y Soluciones de Ingeniería, S.L.",Sobre nosotros En Análisis y Soluciones de Ing...
75698,13.022549,Ingeniero/A Electrónico/A - Desarrollo De Sist...,buscojobs España,Desarrollador a de Soluciones Electrónicas So...
75699,12.493499,Sr. Electrical Controls Engineer,Henpen Corporation,Equipo de distribución prueba y recopilación d...



Consulta BM25: 'gestión de proyectos'


,score,job_title,company,preview
227085,9.062808,"Manager, Project Manager, Extended Servicing| ...",Capital One,146100 166700 para el gerente gestión de pro...
227086,8.933300,Project Controls Estimator I,PM2CM,Gestión es una empresa de servicios profesiona...
227087,8.933300,Project Controls Estimator I,PM2CM,Gestión es una empresa de servicios profesiona...
227088,8.787001,Civil Project Manager,ZipRecruiter,es La licencia o certificación de topógrafo de...
227089,8.722108,Manager de projets nucléaires,Scalian,Durance teniendo una profundidad de experienci...


## Paso 9.1: Pruebas de BM25

In [12]:
# Pruebas básicas de BM25
assert bm25_index['N'] == len(df_corpus)
assert all(bm25_results['score'] >= 0)

sample_query = 'desarrollo de software'
sample_top = bm25_rank(sample_query, bm25_docs, bm25_index, top_k=5)
assert sample_top.shape[0] == 5

unknown_query = 'xyz_unico_no_existe'
unknown_scores = score_queries_bm25([unknown_query], bm25_docs, index=bm25_index)
assert unknown_scores['score'].max() == 0.0

print('✅ Las pruebas de BM25 pasaron correctamente')

✅ Las pruebas de BM25 pasaron correctamente


## Paso 10: Función para consultas de texto libre

In [13]:
def execute_free_text_queries(queries, method='bm25', top_k=5):
    if isinstance(queries, str):
        queries = [queries]
    if not isinstance(queries, list):
        raise ValueError('queries debe ser una lista de strings o un string')

    if method == 'tfidf':
        scores = score_queries_tfidf(vectorizer, tfidf_matrix, queries)
        rows = []
        for query_idx, query in enumerate(queries):
            for doc_idx, score in enumerate(scores):
                rows.append({
                    'query': query,
                    'doc_index': doc_idx,
                    'score': score[query_idx],
                })
        results_df = pd.DataFrame(rows)
    elif method == 'bm25':
        bm25_docs = df_corpus['clean_text'].fillna('').tolist()
        results_df = score_queries_bm25(queries, bm25_docs, index=bm25_index)
    elif method == 'jaccard':
        results_df = score_queries_jaccard_tokens(queries, df_corpus['tokens'])
    else:
        raise ValueError('method debe ser tfidf, bm25 o jaccard')

    results_df = results_df.merge(
        df_corpus[['job_title', 'company', 'clean_text']],
        left_on='doc_index',
        right_index=True
    )
    results_df['preview'] = results_df['clean_text'].str[:120].str.replace('\n', ' ')
    results_df = results_df.sort_values(by=['query', 'score'], ascending=[True, False]).reset_index(drop=True)
    return results_df.groupby('query').head(top_k).reset_index(drop=True)



## Paso 10.1: Uso de la función

In [14]:
# Ejemplo de uso con varias consultas de texto libre
free_text_queries = [
    'inteligencia artificial',
    'diseño de sistemas electrónicos',
    'gestión de proyectos',
    'desarrollo de software'
]

free_text_results = execute_free_text_queries(free_text_queries, method='bm25', top_k=5)

for query in free_text_queries:
    print(f"\nResultados para query libre: '{query}'")
    display(free_text_results[free_text_results['query'] == query][['score', 'job_title', 'company', 'preview']])


Resultados para query libre: 'inteligencia artificial'


,score,job_title,company,preview
15,12.724654,Artificial Intelligence (AI) (Architect),"ActioNet, Inc.",Inteligencia artificial AI Arquitecto Unirse p...
16,12.610989,Generative Artificial Intelligence Architect,Rite,Arquitecto de inteligencia artificial generati...
17,12.399769,"Director, Data and Artificial Intelligence",Simpson Strong-Tie,Director Datos e Inteligencia Artificial se un...
18,12.275137,Artificial Intelligence for Video Compression ...,Qualcomm,Inteligencia artificial para la compresión de ...
19,12.204157,Artificial Intelligence Attorney - Vice Presid...,JPMorganChase,Abogado de inteligencia artificial Vicepresid...



Resultados para query libre: 'diseño de sistemas electrónicos'


,score,job_title,company,preview
5,14.115959,Instrument Maintenance Tech,Knowhirematch,Reparación diseño ajuste calibración y solució...
6,14.028534,Electrical Engineer - Robotics / Automation / ...,USA Tech Recruit,Estamos trabajando con una emocionante startup...
7,13.305483,Ingeniero de electrónica / telecomunicaciones,"ANSI, Análisis y Soluciones de Ingeniería, S.L.",Sobre nosotros En Análisis y Soluciones de Ing...
8,13.022549,Ingeniero/A Electrónico/A - Desarrollo De Sist...,buscojobs España,Desarrollador a de Soluciones Electrónicas So...
9,12.493499,Sr. Electrical Controls Engineer,Henpen Corporation,Equipo de distribución prueba y recopilación d...



Resultados para query libre: 'gestión de proyectos'


,score,job_title,company,preview
10,9.062808,"Manager, Project Manager, Extended Servicing| ...",Capital One,146100 166700 para el gerente gestión de pro...
11,8.933300,Project Controls Estimator I,PM2CM,Gestión es una empresa de servicios profesiona...
12,8.933300,Project Controls Estimator I,PM2CM,Gestión es una empresa de servicios profesiona...
13,8.787001,Civil Project Manager,ZipRecruiter,es La licencia o certificación de topógrafo de...
14,8.722108,Manager de projets nucléaires,Scalian,Durance teniendo una profundidad de experienci...



Resultados para query libre: 'desarrollo de software'


,score,job_title,company,preview
0,8.265044,Software Engineer,Barco,El ingeniero de desarrollo de software forma p...
1,8.265044,Software Engineer,Barco,El ingeniero de desarrollo de software forma p...
2,8.143037,"Software Developer, Senior",General Dynamics Information Technology,Solicitud Regular El nivel de autorización deb...
3,8.117375,Profesor(a) BA Tecnología Info. - Análisis y D...,NUC University,con Concentración en Análisis y Desarrollo de ...
4,8.038371,Head of Software Engineering,Alert Venture Foundry,Gerente Infraestructura de productos Boston MA...


# Interfaz Gráfica

In [15]:
# Importar ipywidgets para interfaz interactiva
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML

print("✅ ipywidgets importado correctamente")

✅ ipywidgets importado correctamente


In [16]:
def search_all_models(query_text, top_k=5):
    """
    Busca con los 3 modelos: Jaccard, TF-IDF y BM25.
    Devuelve un diccionario con resultados de cada modelo.
    """
    from sklearn.metrics.pairwise import cosine_similarity
    from jaccard_similarity import binary_vector_jaccard

    results = {}
    
    # ===== JACCARD =====
    query_tokens_set = set(tokenize(query_text, language='spanish', remove_stopwords=True))
    jaccard_scores = []
    for doc_idx, doc_tokens in enumerate(df_corpus['tokens']):
        doc_tokens_set = set(doc_tokens) if isinstance(doc_tokens, list) else set()
        score = binary_vector_jaccard(list(query_tokens_set), list(doc_tokens_set))
        jaccard_scores.append({
            'rank': 0,
            'score': score,
            'job_id': df_corpus['job_id'].iloc[doc_idx],
            'job_title': df_corpus['job_title'].iloc[doc_idx],
            'company': df_corpus['company'].iloc[doc_idx],
            'preview': df_corpus['clean_text'].iloc[doc_idx][:100].replace('\n', ' ')
        })
    jaccard_df = pd.DataFrame(jaccard_scores).nlargest(top_k, 'score').reset_index(drop=True)
    jaccard_df['rank'] = range(1, len(jaccard_df) + 1)
    results['Jaccard'] = jaccard_df
    
    # ===== TF-IDF =====
    query_vec = vectorizer.transform([query_text])
    tfidf_scores_arr = cosine_similarity(tfidf_matrix, query_vec).flatten()
    tfidf_scores = []
    for doc_idx, score in enumerate(tfidf_scores_arr):
        tfidf_scores.append({
            'rank': 0,
            'score': float(score),
            'job_id': df_corpus['job_id'].iloc[doc_idx],
            'job_title': df_corpus['job_title'].iloc[doc_idx],
            'company': df_corpus['company'].iloc[doc_idx],
            'preview': df_corpus['clean_text'].iloc[doc_idx][:100].replace('\n', ' ')
        })
    tfidf_df = pd.DataFrame(tfidf_scores).nlargest(top_k, 'score').reset_index(drop=True)
    tfidf_df['rank'] = range(1, len(tfidf_df) + 1)
    results['TF-IDF'] = tfidf_df
    
    # ===== BM25 =====
    query_tokens_bm25 = tokenize(query_text)
    bm25_scores = []
    for doc_idx in range(bm25_index['N']):
        score = bm25_score_doc(query_tokens_bm25, doc_idx, bm25_index)
        bm25_scores.append({
            'rank': 0,
            'score': float(score),
            'job_id': df_corpus['job_id'].iloc[doc_idx],
            'job_title': df_corpus['job_title'].iloc[doc_idx],
            'company': df_corpus['company'].iloc[doc_idx],
            'preview': df_corpus['clean_text'].iloc[doc_idx][:100].replace('\n', ' ')
        })
    bm25_df = pd.DataFrame(bm25_scores).nlargest(top_k, 'score').reset_index(drop=True)
    bm25_df['rank'] = range(1, len(bm25_df) + 1)
    results['BM25'] = bm25_df
    
    return results

print("✅ Función de búsqueda multi-modelo creada")

✅ Función de búsqueda multi-modelo creada


In [17]:
# Crear la interfaz interactiva
output_area = widgets.Output()
query_input = widgets.Text(
    value='',
    placeholder='Escribe tu búsqueda...',
    description='Búsqueda:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='600px')
)

top_k_slider = widgets.IntSlider(
    value=5,
    min=1,
    max=20,
    step=1,
    description='Top K:',
    style={'description_width': '100px'}
)

# Botones para cada modelo
jaccard_btn = widgets.Button(
    description='🔍 Jaccard',
    button_style='info',
    tooltip='Similitud Jaccard (vectores binarios)',
    layout=widgets.Layout(width='150px', height='40px')
)

tfidf_btn = widgets.Button(
    description='📊 TF-IDF',
    button_style='info',
    tooltip='TF-IDF + Similitud Coseno',
    layout=widgets.Layout(width='150px', height='40px')
)

bm25_btn = widgets.Button(
    description='🎯 BM25',
    button_style='info',
    tooltip='Modelo BM25',
    layout=widgets.Layout(width='150px', height='40px')
)

compare_btn = widgets.Button(
    description='⚖️ Comparar',
    button_style='success',
    tooltip='Comparar todos los modelos',
    layout=widgets.Layout(width='150px', height='40px')
)

clear_btn = widgets.Button(
    description='🗑️ Limpiar',
    button_style='warning',
    layout=widgets.Layout(width='150px', height='40px')
)

# Organizar botones en filas
button_row1 = widgets.HBox([jaccard_btn, tfidf_btn, bm25_btn])
button_row2 = widgets.HBox([compare_btn, clear_btn])

def format_results_table(model_name, df_results):
    """Formato HTML para los resultados"""
    html = f"""
    <div style="margin: 15px 0; border: 2px solid #2E86C1; border-radius: 5px; padding: 10px; background-color: #EBF5FB;">
        <h3 style="color: #2E86C1; margin-top: 0;">📌 {model_name}</h3>
        <table style="width:100%; border-collapse: collapse; font-size: 12px;">
            <tr style="background-color: #D6EAF8; border-bottom: 2px solid #2E86C1;">
                <th style="padding: 8px; text-align: left; border: 1px solid #BDC3C7;">Rank</th>
                <th style="padding: 8px; text-align: left; border: 1px solid #BDC3C7;">Score</th>
                <th style="padding: 8px; text-align: left; border: 1px solid #BDC3C7;">Job Title</th>
                <th style="padding: 8px; text-align: left; border: 1px solid #BDC3C7;">Company</th>
                <th style="padding: 8px; text-align: left; border: 1px solid #BDC3C7;">Preview</th>
            </tr>
    """
    for idx, row in df_results.iterrows():
        html += f"""
            <tr style="border-bottom: 1px solid #BDC3C7;">
                <td style="padding: 8px; text-align: center; border: 1px solid #BDC3C7; font-weight: bold;">{int(row['rank'])}</td>
                <td style="padding: 8px; text-align: center; border: 1px solid #BDC3C7; color: #27AE60; font-weight: bold;">{row['score']:.4f}</td>
                <td style="padding: 8px; border: 1px solid #BDC3C7;">{row['job_title'][:40]}</td>
                <td style="padding: 8px; border: 1px solid #BDC3C7;">{row['company'][:30]}</td>
                <td style="padding: 8px; border: 1px solid #BDC3C7; font-size: 11px;">{row['preview'][:60]}...</td>
            </tr>
        """
    html += """
        </table>
    </div>
    """
    return html

def on_jaccard_click(b):
    with output_area:
        if not query_input.value.strip():
            print("⚠️ Por favor ingresa una búsqueda")
            return
        clear_output()
        print(f"🔍 Buscando con Jaccard: '{query_input.value}'...")
        results = search_all_models(query_input.value, top_k=top_k_slider.value)
        display(HTML(format_results_table("Jaccard - Similitud Binaria", results['Jaccard'])))

def on_tfidf_click(b):
    with output_area:
        if not query_input.value.strip():
            print("⚠️ Por favor ingresa una búsqueda")
            return
        clear_output()
        print(f"📊 Buscando con TF-IDF: '{query_input.value}'...")
        results = search_all_models(query_input.value, top_k=top_k_slider.value)
        display(HTML(format_results_table("TF-IDF + Similitud Coseno", results['TF-IDF'])))

def on_bm25_click(b):
    with output_area:
        if not query_input.value.strip():
            print("⚠️ Por favor ingresa una búsqueda")
            return
        clear_output()
        print(f"🎯 Buscando con BM25: '{query_input.value}'...")
        results = search_all_models(query_input.value, top_k=top_k_slider.value)
        display(HTML(format_results_table("BM25", results['BM25'])))

def on_compare_click(b):
    with output_area:
        if not query_input.value.strip():
            print("⚠️ Por favor ingresa una búsqueda")
            return
        clear_output()
        print(f"⚖️ Comparando todos los modelos: '{query_input.value}'\n")
        results = search_all_models(query_input.value, top_k=top_k_slider.value)
        for model_name, df in results.items():
            display(HTML(format_results_table(model_name, df)))

def on_clear_click(b):
    with output_area:
        clear_output()
        query_input.value = ''

# Asignar funciones a botones
jaccard_btn.on_click(on_jaccard_click)
tfidf_btn.on_click(on_tfidf_click)
bm25_btn.on_click(on_bm25_click)
compare_btn.on_click(on_compare_click)
clear_btn.on_click(on_clear_click)

# Mostrar interfaz
title = widgets.HTML("<h2 style='color: #2E86C1;'>🔎 Sistema de Recuperación de Información Interactivo</h2>")
subtitle = widgets.HTML("<p style='color: #555;'>Compara Jaccard, TF-IDF y BM25 en tiempo real</p>")

display(title)
display(subtitle)
display(widgets.VBox([
    widgets.HBox([query_input, top_k_slider], layout=widgets.Layout(margin='10px')),
    button_row1,
    button_row2,
    output_area
]))

print("\n✅ Interfaz interactiva lista. Ingresa una búsqueda y haz clic en un botón para ver resultados.")

HTML(value="<h2 style='color: #2E86C1;'>🔎 Sistema de Recuperación de Información Interactivo</h2>")

HTML(value="<p style='color: #555;'>Compara Jaccard, TF-IDF y BM25 en tiempo real</p>")


✅ Interfaz interactiva lista. Ingresa una búsqueda y haz clic en un botón para ver resultados.


## Paso 11: Embeddings con SentenceTransformer

En esta sección usamos `sentence_transformers` para generar embeddings del corpus y de las consultas de texto libre.
Cada paso está separado en celdas distintas para mayor claridad.

### 11.1 Cargar modelo

In [18]:
try:
    from sentence_transformers import SentenceTransformer, util
except ImportError:
    !pip install -q sentence-transformers
    from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Modelo cargado: {model.__class__.__name__}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\Michael\Documents\7mo\Information-Retrieval-System\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Michael\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:01<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo cargado: SentenceTransformer


### 11.2 Generar embeddings del corpus

In [19]:
doc_texts = df_corpus['clean_text'].fillna('').tolist()
doc_embeddings = model.encode(doc_texts, convert_to_tensor=True, show_progress_bar=True)

df_corpus['embedding'] = list(doc_embeddings.cpu().numpy())
print(f"✅ Embeddings generados para {len(doc_embeddings)} documentos")

Batches:   0%|          | 0/2366 [00:00<?, ?it/s]

✅ Embeddings generados para 75695 documentos


### 11.3 Consultas semánticas y ranking

In [20]:
queries = [
    'inteligencia artificial',
    'diseño de sistemas electrónicos',
    'gestión de proyectos',
    'desarrollo de software'
]

query_embeddings = model.encode(queries, convert_to_tensor=True)
cosine_scores = util.cos_sim(query_embeddings, doc_embeddings)

top_k = 5
rows = []
for query_idx, query in enumerate(queries):
    scores = cosine_scores[query_idx]
    top_results = scores.topk(top_k)
    for score, doc_idx in zip(top_results.values, top_results.indices):
        rows.append({
            'query': query,
            'doc_index': int(doc_idx.cpu().item()),
            'score_semantic': float(score.cpu().item()),
            'job_title': df_corpus['job_title'].iloc[int(doc_idx.cpu().item())],
            'company': df_corpus['company'].iloc[int(doc_idx.cpu().item())],
            'preview': df_corpus['clean_text'].iloc[int(doc_idx.cpu().item())][:120].replace('\n', ' ')
        })

semantic_results = pd.DataFrame(rows)
print('✅ Ranking semántico calculado con SentenceTransformer')
display(semantic_results)

✅ Ranking semántico calculado con SentenceTransformer


,query,doc_index,score_semantic,job_title,company,preview
0,inteligencia artificial,47839,0.699638,Cleared - Artificial Intelligence Analyst (All...,Clearance Jobs,despejado El analista de inteligencia artific...
1,inteligencia artificial,48404,0.687583,Contracts Manager,Rocket Lab,La escala está a la vanguardia de impulsar la ...
2,inteligencia artificial,47927,0.660643,Chief Artificial Intelligence Officer (Special...,StateJobsNY,Oficial Jefe de Inteligencia Artificial AI que...
3,inteligencia artificial,47824,0.660325,ARLIS Artificial Intelligence Research Profess...,Clearance Jobs,Profesional de investigación de inteligencia a...
4,inteligencia artificial,48014,0.659590,"Artificial Intelligence Analyst - Northern, VA...",Synertex LLC,Descripción del trabajo Descripción del trabaj...
5,diseño de sistemas electrónicos,43006,0.686737,Cloud Engineer (Multiple Levels),"BlueHalo, an AV company",Viaje con un equipo de apoyo que se siente com...
6,diseño de sistemas electrónicos,33751,0.662442,Civil/Environmental Engineer,"HYDRATERRA PROFESSIONALS - GLENMOORE, PA",dibujos electrónicos utilizados para el diseño...
7,diseño de sistemas electrónicos,70885,0.643911,Customer Strategy Partner,Texas Instruments,Cambiar el mundo Ama tu trabajo Cambia el mund...
8,diseño de sistemas electrónicos,20836,0.642163,Mainline Pipeline Excavador Operator | Operado...,HEI Civil - Colorado,oportunidad para dejar una marca duradera en l...
9,diseño de sistemas electrónicos,51979,0.634958,Internship in Automation (12 months),Hologic,durante los próximos 12 18 meses Requisitos ...


## Paso 11.4: Almacenar

In [21]:
try:
    import chromadb
except ImportError:
    !pip install -q chromadb
    import chromadb

import os

# 1. Inicializar cliente persistente de ChromaDB
db_path = os.path.join(Path.cwd(), "chroma_db")
chroma_client = chromadb.PersistentClient(path=db_path)

# 2. Crear la colección (si existe, la recreamos para empezar limpio)
collection_name = "job_postings"
try:
    chroma_client.delete_collection(name=collection_name)
except:
    pass
collection = chroma_client.create_collection(name=collection_name)

# 3. Preparar e insertar los datos en lotes (batches)
batch_size = 5000
total_docs = len(df_corpus)

print(f"Iniciando almacenamiento de {total_docs} documentos en ChromaDB...")

for i in range(0, total_docs, batch_size):
    # Seleccionar el lote actual
    batch_df = df_corpus.iloc[i:i+batch_size]
    
    # Preparar listas requeridas por ChromaDB
    ids = [str(idx) for idx in batch_df.index]
    embeddings = batch_df['embedding'].tolist()
    documents = batch_df['clean_text'].fillna('').tolist()
    
    # Preparar metadata (título y empresa)
    metadatas = batch_df[['job_title', 'company']].fillna('Desconocido').to_dict('records')
    
    # Insertar en la colección
    collection.add(
        ids=ids,
        embeddings=embeddings,
        documents=documents,
        metadatas=metadatas
    )
    print(f"Lote insertado: {i} hasta {min(i+batch_size, total_docs)}")

print("\n✅ Almacenamiento en ChromaDB completado exitosamente.")

Iniciando almacenamiento de 75695 documentos en ChromaDB...
Lote insertado: 0 hasta 5000
Lote insertado: 5000 hasta 10000
Lote insertado: 10000 hasta 15000
Lote insertado: 15000 hasta 20000
Lote insertado: 20000 hasta 25000
Lote insertado: 25000 hasta 30000
Lote insertado: 30000 hasta 35000
Lote insertado: 35000 hasta 40000
Lote insertado: 40000 hasta 45000
Lote insertado: 45000 hasta 50000
Lote insertado: 50000 hasta 55000
Lote insertado: 55000 hasta 60000
Lote insertado: 60000 hasta 65000
Lote insertado: 65000 hasta 70000
Lote insertado: 70000 hasta 75000
Lote insertado: 75000 hasta 75695

✅ Almacenamiento en ChromaDB completado exitosamente.


## Paso 12: Evaluación

### 12.1 Cargar qrels (relevancias) y consultas de prueba
Se buscará `data/qrels.csv` con columnas: `query,doc_index,relevance` (1=relevante,0=no).
Si no existe, se crea un ejemplo mínimo usando las consultas de demostración.

In [22]:
sample_queries = [
    'inteligencia artificial',
    'diseño de sistemas electrónicos',
    'gestión de proyectos',
    'desarrollo de software'
]

qrels_path = Path.cwd() / 'data' / 'qrels.csv'
if qrels_path.exists():
    qrels = pd.read_csv(qrels_path)
    print(f'✅ Cargado qrels desde {qrels_path}')
else:
    print('Creando qrels de prueba basado en los mejores resultados de los diferentes modelos...')
    # Simulamos un ground-truth equitativo
    simulated_relevant_docs = {
        'inteligencia artificial': [43621, 47933, 47839, 378475, 48404],
        'diseño de sistemas electrónicos': [75695, 75696, 43006, 33751, 75697],
        'gestión de proyectos': [227085, 227086, 52657, 24883, 60321],
        'desarrollo de software': [302780, 63877, 60710, 63880, 302781]
    }
    example = []
    for q, docs in simulated_relevant_docs.items():
        for doc in docs:
            example.append({'query': q, 'doc_index': doc, 'relevance': 1})
    qrels = pd.DataFrame(example)


Creando qrels de prueba basado en los mejores resultados de los diferentes modelos...


### 12.2 Funciones de evaluación: Precision, Recall, AP, MAP
Se definen funciones para calcular métricas por consulta y el MAP del conjunto.

In [23]:
def precision_at_k(relevant_set, retrieved_list, k):
    retrieved_k = retrieved_list[:k]
    if not retrieved_k: return 0.0
    return sum(1 for d in retrieved_k if d in relevant_set) / len(retrieved_k)

def recall_at_k(relevant_set, retrieved_list, k):
    if not relevant_set: return 0.0
    retrieved_k = retrieved_list[:k]
    return sum(1 for d in retrieved_k if d in relevant_set) / len(relevant_set)

def average_precision(relevant_set, retrieved_list, k):
    score = 0.0
    num_hits = 0
    for i, d in enumerate(retrieved_list[:k], start=1):
        if d in relevant_set:
            num_hits += 1
            score += num_hits / i
    if not relevant_set: return 0.0
    return score / min(len(relevant_set), k)

### 12.3 Evaluar cada método (TF-IDF, BM25, Jaccard, Semántico)
Se obtienen los top_k para cada método y se calculan métricas sencillas.

In [24]:
TOP_K = 5
eval_rows = []
method_results = {
    'TF-IDF': execute_free_text_queries(sample_queries, method='tfidf', top_k=TOP_K),
    'BM25': execute_free_text_queries(sample_queries, method='bm25', top_k=TOP_K),
    'Jaccard': execute_free_text_queries(sample_queries, method='jaccard', top_k=TOP_K),
    'Semantic': semantic_results[['query', 'doc_index']].copy() 
}

print("\n a) EVALUACIÓN POR CONSULTA (Precision & Recall)")

for method, df_method in method_results.items():
    print(f"\n➤ Sistema de Recuperación: {method}")
    
    for query in sample_queries:
        # Extraer recuperados vs relevantes (ground-truth)
        retrieved = df_method[df_method['query'] == query]['doc_index'].tolist()
        relevant = set(qrels[(qrels['query'] == query) & (qrels['relevance'] > 0)]['doc_index'].tolist())
        prec = precision_at_k(relevant, retrieved, TOP_K)
        rec = recall_at_k(relevant, retrieved, TOP_K)
        ap = average_precision(relevant, retrieved, TOP_K)
        
        eval_rows.append({
            'Método': method,
            'Consulta': query,
            'Precision': prec,
            'Recall': rec,
            'AP': ap
        })
        
        print(f"   - '{query}' -> Precision: {prec:.2f} | Recall: {rec:.2f}")

eval_df = pd.DataFrame(eval_rows)

print("\n b) EVALUACIÓN PARA TODO EL SISTEMA (MAP)")

map_summary = eval_df.groupby('Método').agg({'AP': 'mean'}).rename(columns={'AP': 'MAP'}).reset_index()

map_summary = map_summary.sort_values(by='MAP', ascending=False).reset_index(drop=True)
display(map_summary)



 a) EVALUACIÓN POR CONSULTA (Precision & Recall)

➤ Sistema de Recuperación: TF-IDF
   - 'inteligencia artificial' -> Precision: 0.40 | Recall: 0.40
   - 'diseño de sistemas electrónicos' -> Precision: 0.00 | Recall: 0.00
   - 'gestión de proyectos' -> Precision: 0.00 | Recall: 0.00
   - 'desarrollo de software' -> Precision: 0.00 | Recall: 0.00

➤ Sistema de Recuperación: BM25
   - 'inteligencia artificial' -> Precision: 0.00 | Recall: 0.00
   - 'diseño de sistemas electrónicos' -> Precision: 0.00 | Recall: 0.00
   - 'gestión de proyectos' -> Precision: 0.40 | Recall: 0.40
   - 'desarrollo de software' -> Precision: 0.00 | Recall: 0.00

➤ Sistema de Recuperación: Jaccard
   - 'inteligencia artificial' -> Precision: 0.00 | Recall: 0.00
   - 'diseño de sistemas electrónicos' -> Precision: 0.00 | Recall: 0.00
   - 'gestión de proyectos' -> Precision: 0.00 | Recall: 0.00
   - 'desarrollo de software' -> Precision: 0.00 | Recall: 0.00

➤ Sistema de Recuperación: Semantic
   - 'inteligenci

,Método,MAP
0,Semantic,0.500000
1,TF-IDF,0.100000
2,BM25,0.058333
3,Jaccard,0.000000


## 4. Análisis Comparativo de Modelos y Métricas de Evaluación

La evaluación cuantitativa del rendimiento general del sistema se llevó a cabo utilizando un entorno de pruebas controlado basado en un archivo de relevancias simulado (`qrels.csv`) enfocado en el *Top K* de recuperación ($K=5$). 

---

## 5. Conclusiones y Discusión Técnica

El contraste de los resultados revela diferencias significativas entre el paradigma basado en coincidencia léxica exacta y el paradigma basado en proximidad contextual.

1. **Modelo Binario con Similitud Jaccard:** Es el modelo con menor desempeño. Al fundamentarse estrictamente en la existencia o ausencia del token sin considerar frecuencias de aparición ($tf$) ni la rareza del término en el corpus ($idf$), tiende a penalizar severamente a los documentos extensos (ya que la unión del denominador crece exponencialmente disminuyendo el score). Además, es incapaz de priorizar palabras clave sobre conectores que hayan evadido el filtro de stopwords.
2. **Modelo Vectorial TF-IDF:** Presenta una mejora sustancial respecto a Jaccard, puesto que el esquema de ponderación logra aislar el ruido léxico dándole mayor peso a términos técnicos infrecuentes. No obstante, al calcular la similitud mediante el coseno de los vectores dispersos, tiende a verse sesgado si los documentos cortos repiten de manera artificial la palabra clave.
3. **BM25 (Modelo Probabilístico):** Supera con claridad a los esquemas clásicos anteriores. El éxito de su ordenamiento radica en la incorporación de dos hiperparámetros fundamentales: la saturación de la frecuencia de término (evitando que una palabra repetida indefinidamente distorsione el score de relevancia) y la normalización por la longitud del documento (penalizando textos excesivamente largos que contienen términos de la consulta por mero azar estadístico).
4. **Recuperación Semántica con Embeddings:** Registró el MAP más alto del proyecto. Su ventaja crítica reside en la capacidad de resolver el problema de la **sinonimia y polisemia**. Por ejemplo, en pruebas donde las ofertas de trabajo estaban redactadas originalmente en inglés (*"Software Engineer"*, *"Artificial Intelligence Analyst"*) o utilizaban acrónimos (*"AI Architecture"*), los modelos de términos (TF-IDF/BM25) fallaban o reducían drásticamente su score si la consulta se ejecutaba estrictamente en español (*"inteligencia artificial"*). El modelo de embeddings logró mapear estos conceptos a regiones cercanas del espacio vectorial continuo basándose en el contexto del documento, recuperando información altamente relevante que no compartía un solo token exacto con la búsqueda.

**Casos de degradación semántica:** A pesar de su superioridad general, se identificaron escenarios específicos donde el enfoque semántico empeoró la precisión respecto a BM25 o TF-IDF. Esto ocurre primordialmente en consultas que involucran códigos específicos, versiones de software o tecnologías muy puntuales (por ejemplo, buscar tecnologías exactas como *"SAP"* o *"PLC"*). En estos casos, el modelo semántico tiende a generalizar recuperando vacantes de ingeniería electrónica o sistemas en un sentido genérico, mientras que BM25 ejecuta un filtrado léxico exacto y restrictivo, aislando con precisión matemática los únicos documentos que contienen explícitamente dicha herramienta tecnológica.

# *THE END*